# Data Manipulation — Advanced
## Feature Engineering

**Philosophy:** This note starts where cleaning and reshaping end — data is clean and in the right shape. Each section covers how to create new features for ML/modeling, with explicit attention to *when to use each technique* and *what can go wrong (especially leakage)*.

---

## Decision Table

| If your column is... | Go to |
| :--- | :--- |
| Categorical / string that needs to be numeric | §1 — Encoding Categoricals |
| Numeric but skewed, on different scales, or needs normalization | §2 — Numerical Transformations |
| Continuous numeric that should be grouped into ranges | §3 — Binning & Discretization |
| A datetime that should produce predictive features | §4 — Date & Time Features |
| A time-ordered numeric where past values predict future | §5 — Lag & Window Features |
| Two or more columns whose combination may be informative | §6 — Interaction & Ratio Features |
| Free text in a tabular dataset | §7 — Text as Features |
| Suspiciously good model performance or target in features | §8 — Leakage & Validation |

---
## §1 — Encoding Categorical Variables

Most models require numeric input. The right encoding depends on cardinality, whether order exists, and whether you're using a tree-based or linear model.

In [ ]:
import pandas as pd
import numpy as np

# ── Encoding 1: One-Hot Encoding ─────────────────────────────────────────────
# Use when: nominal (no order), low cardinality (<15 unique), linear models or NNs
dummies = pd.get_dummies(df['platform'], prefix='platform', drop_first=True)
# drop_first=True removes one category to avoid multicollinearity (dummy variable trap)
# For tree-based models, drop_first=False is fine
df = pd.concat([df.drop(columns='platform'), dummies], axis=1)

# Multiple columns at once
df = pd.get_dummies(df, columns=['platform', 'region'], drop_first=True)

# ── Encoding 2: Label Encoding ───────────────────────────────────────────────
# Use when: ordinal (has natural order) — e.g. Low / Medium / High
# Do NOT use for nominal categories with tree-free models — implies false ordering
order_map = {'Low': 0, 'Medium': 1, 'High': 2, 'Very High': 3}
df['risk_level_enc'] = df['risk_level'].map(order_map)

# Using pandas Categorical with defined ordering (remembers the order)
df['risk_level'] = pd.Categorical(
    df['risk_level'],
    categories=['Low', 'Medium', 'High', 'Very High'],
    ordered=True
)
df['risk_level_enc'] = df['risk_level'].cat.codes  # 0, 1, 2, 3

# ── Encoding 3: Frequency Encoding ──────────────────────────────────────────
# Use when: high cardinality, tree-based model, frequency itself is informative
# Replaces each category with how often it appears in the training set
freq_map = df_train['city'].value_counts(normalize=True)   # compute on train only
df['city_freq'] = df['city'].map(freq_map)

# ── Encoding 4: Target Encoding ──────────────────────────────────────────────
# Use when: high cardinality, strong relationship between category and target
# MAJOR LEAKAGE RISK — must be done correctly with cross-validation
# Safe approach: compute on train set only, using out-of-fold means
from sklearn.model_selection import KFold

def target_encode_cv(df_train, df_test, col, target, n_splits=5):
    kf = KFold(n_splits=n_splits, shuffle=True, random_state=42)
    train_enc = np.zeros(len(df_train))
    global_mean = df_train[target].mean()

    for tr_idx, val_idx in kf.split(df_train):
        means = df_train.iloc[tr_idx].groupby(col)[target].mean()
        train_enc[val_idx] = df_train.iloc[val_idx][col].map(means).fillna(global_mean)

    # Apply to test: use full train set means
    train_means = df_train.groupby(col)[target].mean()
    test_enc = df_test[col].map(train_means).fillna(global_mean)
    return train_enc, test_enc

df_train['city_target'], df_test['city_target'] = target_encode_cv(
    df_train, df_test, col='city', target='churn'
)

**Encoding selection guide:**

| Situation | Encoding | Why |
| :--- | :--- | :--- |
| Nominal, low cardinality (<15), linear model | One-hot | No implied order |
| Ordinal (Low/Med/High) | Label / Ordinal | Order is meaningful |
| High cardinality (>50), tree model | Frequency | Avoids sparse columns |
| High cardinality, strong target signal | Target (with CV) | Most predictive — but leakage risk |

**Common mistakes:**
- Label-encoding a nominal column (e.g. `city`) for a linear model — implies `Paris > London > Tokyo` which is meaningless and distorts coefficients
- Forgetting `drop_first=True` in `get_dummies` for linear models — one column is perfectly predictable from the others (dummy variable trap)
- Computing target encoding on the full training set without CV — your model sees the target in its features, causing massive overfitting

---
## §2 — Numerical Transformations

Scaling and transforming numeric features matters for distance-based models (logistic regression, SVM, KNN, neural nets) but not for tree-based models (XGBoost, Random Forest). Always fit scalers on train, apply to test.

In [ ]:
from sklearn.preprocessing import StandardScaler, MinMaxScaler, RobustScaler, PowerTransformer

# ── Transform 1: Log transform — for right-skewed distributions ──────────────
# Use when: revenue, income, counts — anything with a long right tail
df['log_revenue'] = np.log1p(df['revenue'])    # log(x+1): handles x=0 safely
# When NOT to use: symmetric distributions, negative values, already normal

# ── Transform 2: Standardization (Z-score) ──────────────────────────────────
# Result: mean=0, std=1. Use for: linear models, SVM, KNN, neural nets
scaler = StandardScaler()
X_train[['age', 'income']] = scaler.fit_transform(X_train[['age', 'income']])
X_test[['age', 'income']]  = scaler.transform(X_test[['age', 'income']])  # use train fit

# ── Transform 3: Min-Max Scaling ────────────────────────────────────────────
# Result: all values in [0, 1]. Use for: neural nets, image pixel values
# Sensitive to outliers — one extreme value compresses everything else
scaler = MinMaxScaler()
X_train[['score']] = scaler.fit_transform(X_train[['score']])
X_test[['score']]  = scaler.transform(X_test[['score']])

# ── Transform 4: Robust Scaling ──────────────────────────────────────────────
# Uses median and IQR — not affected by outliers
# Use when: data has outliers you want to keep but not let dominate
scaler = RobustScaler()
X_train[['revenue']] = scaler.fit_transform(X_train[['revenue']])

# ── Transform 5: Box-Cox / Yeo-Johnson ──────────────────────────────────────
# Automatically finds the best power transform to make distribution normal
# Box-Cox: requires positive values only
# Yeo-Johnson: works with zero and negative values
pt = PowerTransformer(method='yeo-johnson')
X_train[['revenue']] = pt.fit_transform(X_train[['revenue']])
X_test[['revenue']]  = pt.transform(X_test[['revenue']])

# ── Quick check: plot before and after ───────────────────────────────────────
import matplotlib.pyplot as plt
fig, axes = plt.subplots(1, 2, figsize=(10, 3))
df['revenue'].hist(ax=axes[0], bins=50); axes[0].set_title('Raw')
df['log_revenue'].hist(ax=axes[1], bins=50); axes[1].set_title('Log-transformed')
plt.tight_layout(); plt.show()

**Transformation selection guide:**

| Situation | Transformation |
| :--- | :--- |
| Right-skewed (revenue, counts) | `np.log1p()` |
| Linear/logistic regression, SVM, KNN | StandardScaler |
| Neural network inputs | MinMaxScaler or StandardScaler |
| Data has significant outliers | RobustScaler |
| Want to auto-normalize any distribution | PowerTransformer (Yeo-Johnson) |
| Tree-based model (XGBoost, RF) | None required |

**Common mistakes:**
- Fitting the scaler on the full dataset before splitting — leaks test set distribution into the scaler; always fit on train only
- Scaling the target variable when it isn't needed — scaling `y` for regression is usually unnecessary and complicates interpretation
- Using `np.log()` on columns with zeros — produces `-inf`; always use `np.log1p()` for safety

---
## §3 — Binning & Discretization

Convert a continuous variable into labeled groups. Useful when the relationship with the target is non-linear and step-like, or when business rules define meaningful ranges.

In [ ]:
# ── pd.cut — equal-width bins ────────────────────────────────────────────────
# Use when: bin edges come from domain knowledge or business rules
df['age_group'] = pd.cut(
    df['age'],
    bins   = [0, 18, 35, 55, 100],
    labels = ['Under 18', '18-34', '35-54', '55+'],
    right  = True,    # intervals are (left, right] by default
    include_lowest = True  # include the leftmost edge
)

# ── pd.qcut — equal-frequency bins (quantile-based) ─────────────────────────
# Use when: you want roughly equal-sized groups (e.g. quartiles)
# Important: compute quantiles on train set only
df_train['revenue_quartile'] = pd.qcut(
    df_train['revenue'],
    q      = 4,
    labels = ['Q1', 'Q2', 'Q3', 'Q4'],
    duplicates = 'drop'  # handles duplicate bin edges (common with sparse data)
)

# Apply train bins to test (use retbins=True to get the bin edges)
_, bin_edges = pd.qcut(df_train['revenue'], q=4, retbins=True, duplicates='drop')
df_test['revenue_quartile'] = pd.cut(
    df_test['revenue'],
    bins   = bin_edges,
    labels = ['Q1', 'Q2', 'Q3', 'Q4'],
    include_lowest = True
)

# ── Custom business-logic bins ────────────────────────────────────────────────
# Use when: business definitions dictate the thresholds
def segment_user(row):
    if row['revenue'] >= 1000 and row['tenure_days'] >= 365:
        return 'VIP'
    elif row['revenue'] >= 500:
        return 'Premium'
    elif row['revenue'] > 0:
        return 'Standard'
    else:
        return 'Inactive'

# Vectorized equivalent using np.select (preferred over apply)
conditions = [
    (df['revenue'] >= 1000) & (df['tenure_days'] >= 365),
    df['revenue'] >= 500,
    df['revenue'] > 0
]
choices = ['VIP', 'Premium', 'Standard']
df['user_segment'] = np.select(conditions, choices, default='Inactive')

# ── Check bin distribution after cutting ─────────────────────────────────────
print(df['age_group'].value_counts().sort_index())

**`pd.cut` vs `pd.qcut`:**

| | `pd.cut` | `pd.qcut` |
| :--- | :--- | :--- |
| Bin width | Equal width | Equal frequency |
| Group sizes | Unequal (depends on distribution) | Roughly equal |
| Edges from | You specify | Computed from data |
| Use when | Domain-defined thresholds | Want balanced groups |
| Leakage risk | Low | High — compute on train only |

**Common mistakes:**
- Computing `qcut` on the full dataset — bin edges depend on the distribution, which leaks test set info into the model
- Forgetting `include_lowest=True` — the lowest value falls outside all bins and becomes NaN
- Using `pd.cut` labels as strings directly in a linear model — encode them afterward with one-hot or ordinal encoding

---
## §4 — Date & Time Features

Raw timestamps are not useful to most models. Extract meaningful components — and encode cyclical features correctly so models understand that hour 23 is close to hour 0.

In [ ]:
df['ts'] = pd.to_datetime(df['ts'])

# ── Basic extractions ────────────────────────────────────────────────────────
df['year']        = df['ts'].dt.year
df['month']       = df['ts'].dt.month          # 1–12
df['day']         = df['ts'].dt.day            # 1–31
df['hour']        = df['ts'].dt.hour           # 0–23
df['day_of_week'] = df['ts'].dt.dayofweek      # 0=Mon, 6=Sun
df['is_weekend']  = (df['ts'].dt.dayofweek >= 5).astype(int)
df['quarter']     = df['ts'].dt.quarter        # 1–4
df['week_of_year']= df['ts'].dt.isocalendar().week.astype(int)

# ── Cyclical encoding — critical for hour, month, day_of_week ───────────────
# Problem: raw hour 23 and hour 0 are far apart numerically but close in time
# Solution: encode as (sin, cos) pair — places similar times close in 2D space

def cyclical_encode(series, max_val):
    return (
        np.sin(2 * np.pi * series / max_val),
        np.cos(2 * np.pi * series / max_val)
    )

df['hour_sin'], df['hour_cos']           = cyclical_encode(df['hour'],        24)
df['month_sin'], df['month_cos']         = cyclical_encode(df['month'],       12)
df['dow_sin'], df['dow_cos']             = cyclical_encode(df['day_of_week'],  7)

# ── Elapsed time features ────────────────────────────────────────────────────
reference_date = pd.Timestamp('2020-01-01')
df['days_since_ref']     = (df['ts'] - reference_date).dt.days
df['days_since_signup']  = (df['ts'] - df['signup_date']).dt.days
df['days_to_expiry']     = (df['expiry_date'] - df['ts']).dt.days

# ── Time-since-last-event (per user) ─────────────────────────────────────────
df = df.sort_values(['user_id', 'ts'])
df['prev_ts']           = df.groupby('user_id')['ts'].shift(1)
df['hours_since_last']  = (df['ts'] - df['prev_ts']).dt.total_seconds() / 3600

# ── Business day features ─────────────────────────────────────────────────────
from pandas.tseries.holiday import USFederalHolidayCalendar
cal     = USFederalHolidayCalendar()
holidays = cal.holidays(start=df['ts'].min(), end=df['ts'].max())
df['is_holiday']  = df['ts'].dt.normalize().isin(holidays).astype(int)
df['is_business_day'] = (~df['ts'].dt.weekday.isin([5, 6]) & ~df['is_holiday'].astype(bool)).astype(int)

**When to use cyclical encoding:**

| Feature | Range | Use cyclical? |
| :--- | :--- | :--- |
| Hour of day | 0–23 | ✅ Yes — 23 is close to 0 |
| Day of week | 0–6 | ✅ Yes — Sunday is close to Monday |
| Month | 1–12 | ✅ Yes — December is close to January |
| Year | 2020–2024 | ❌ No — not cyclical |
| Days since event | 0–∞ | ❌ No — not cyclical |

**Common mistakes:**
- Using raw hour as a numeric feature — model sees hour 1 and hour 23 as far apart; use sin/cos encoding instead
- Computing `days_since_signup` without sorting by user and date first — `shift(1)` operates on row order, not time order
- Extracting too many date features — correlated features (month + quarter + season) add noise; pick the most granular

---
## §5 — Lag & Window Features

Use past values to predict future values. The most powerful feature type for time-series and sequential data — and the easiest place to introduce lookahead leakage.

In [ ]:
# Always sort before creating lag or window features
df = df.sort_values(['user_id', 'date']).reset_index(drop=True)

# ── Lag features — past values of the target or other columns ────────────────
# LAG(revenue, 1): revenue from previous period
df['rev_lag1']  = df.groupby('user_id')['revenue'].shift(1)   # 1 period ago
df['rev_lag7']  = df.groupby('user_id')['revenue'].shift(7)   # 7 periods ago
df['rev_lag30'] = df.groupby('user_id')['revenue'].shift(30)  # 30 periods ago

# ── Rolling window features — statistics over the last N rows ─────────────────
# min_periods=1 avoids NaN for the first few rows of each group
df['rev_rolling_mean_7']  = df.groupby('user_id')['revenue'].transform(
    lambda x: x.shift(1).rolling(7,  min_periods=1).mean()   # shift(1) prevents using current row
)
df['rev_rolling_std_7']   = df.groupby('user_id')['revenue'].transform(
    lambda x: x.shift(1).rolling(7,  min_periods=1).std()
)
df['rev_rolling_max_30']  = df.groupby('user_id')['revenue'].transform(
    lambda x: x.shift(1).rolling(30, min_periods=1).max()
)

# ── Expanding window — cumulative from the start of the series ───────────────
# Useful for lifetime statistics (total purchases ever)
df['cumulative_rev'] = df.groupby('user_id')['revenue'].transform(
    lambda x: x.shift(1).expanding().sum()   # sum of all past revenue, not including current
)

# ── Delta features — change from previous period ─────────────────────────────
df['rev_delta']      = df['revenue'] - df['rev_lag1']               # absolute change
df['rev_pct_change'] = df['rev_delta'] / df['rev_lag1'].replace(0, np.nan)  # % change

# ── Safe train/test split for time series ────────────────────────────────────
# NEVER use random split — it causes lookahead leakage
# Use a cutoff date instead
cutoff = pd.Timestamp('2024-01-01')
train  = df[df['date'] <  cutoff]
test   = df[df['date'] >= cutoff]
# Verify: test features should only use data from before the cutoff
# Lag features for test rows should look back into train rows — this is correct
# Rolling features for test rows that include post-cutoff data — this is leakage

**Leakage anatomy for lag features:**

```
Timeline: ... | day 5 | day 6 | day 7 (target) |

lag1 = day 6 revenue    ← OK: past data
lag7 = day 0 revenue    ← OK: past data
rolling_mean without shift(1) = mean(day 5, 6, 7)  ← LEAKAGE: includes target day
rolling_mean with shift(1)    = mean(day 4, 5, 6)  ← OK: excludes target day
```

**Common mistakes:**
- Not calling `.shift(1)` before `.rolling()` — the rolling window includes the current row, which is the target
- Using random train/test split for time series — future rows leak into training via lag features
- Not sorting by date within each group before `.shift()` — shift operates on row position, not time order
- Forgetting `min_periods=1` in `.rolling()` — the first N-1 rows of each group become NaN, losing data

---
## §6 — Interaction & Ratio Features

Combining two or more columns can capture relationships that neither column captures alone. Most valuable when domain knowledge suggests the combination is meaningful.

In [ ]:
# ── Ratio features — normalize one column by another ─────────────────────────
# Use when: the absolute value is less meaningful than the relative value
df['ctr']             = df['clicks'] / df['impressions'].replace(0, np.nan)  # click-through rate
df['conversion_rate'] = df['conversions'] / df['clicks'].replace(0, np.nan)
df['rev_per_session'] = df['revenue'] / df['sessions'].replace(0, np.nan)
df['arpu']            = df['revenue'] / df['users'].replace(0, np.nan)       # avg revenue per user

# ── Difference features — gap between two related values ─────────────────────
df['price_vs_avg']    = df['price'] - df['category_avg_price']   # above/below category mean
df['score_delta']     = df['current_score'] - df['baseline_score']
df['days_to_renewal'] = (df['renewal_date'] - df['today']).dt.days

# ── Product (multiplication) — interaction between two features ───────────────
# Use when: both features together affect the outcome (synergy effect)
df['tenure_x_spend']  = df['tenure_days'] * df['total_spend']    # loyalty × value
df['age_x_income']    = df['age'] * df['income']                 # demographic interaction

# ── Polynomial features — automated interaction and power terms ───────────────
# Use for linear models when you suspect non-linear relationships
from sklearn.preprocessing import PolynomialFeatures
poly = PolynomialFeatures(degree=2, include_bias=False, interaction_only=False)
X_poly = poly.fit_transform(X_train[['age', 'income', 'tenure']])
# Produces: age, income, tenure, age², income², tenure², age×income, age×tenure, income×tenure
# Warning: features grow as O(n²) — only use with a small number of base features

# ── Group-relative features — how does this row compare to its group? ─────────
df['rev_vs_region_avg'] = df['revenue'] - df.groupby('region')['revenue'].transform('mean')
df['rev_pct_of_region'] = df['revenue'] / df.groupby('region')['revenue'].transform('sum')
df['rev_rank_in_region']= df.groupby('region')['revenue'].rank(method='dense', ascending=False)

**Common mistakes:**
- Dividing without replacing zeros — `clicks / impressions` when `impressions = 0` produces `inf` or `NaN`; always `.replace(0, np.nan)` the denominator first
- Creating all possible interactions automatically without domain guidance — most interactions are noise; domain-motivated features outperform automated polynomial features
- Not checking for multicollinearity after adding ratio features — `rev_per_user` and `total_rev` and `user_count` are linearly dependent; use VIF or correlation matrix to diagnose

---
## §7 — Text as Features

For tabular ML problems with one or two text columns — not full NLP, just enough to extract signal from free text.

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer, CountVectorizer

# ── Simple text features — no vectorization needed ───────────────────────────
df['desc_length']    = df['description'].str.len()           # character count
df['desc_word_count']= df['description'].str.split().str.len()  # word count
df['has_promo']      = df['description'].str.lower().str.contains('sale|discount|free', na=False).astype(int)
df['exclamation_ct'] = df['description'].str.count('!')
df['has_url']        = df['description'].str.contains(r'https?://', na=False).astype(int)

# ── TF-IDF — term frequency–inverse document frequency ───────────────────────
# Use when: text column has meaningful vocabulary and medium cardinality
# Fit on train, transform both train and test
tfidf = TfidfVectorizer(
    max_features = 100,        # keep top-100 most informative terms
    ngram_range  = (1, 2),     # include bigrams: 'fast shipping' as one feature
    stop_words   = 'english',  # remove 'the', 'and', 'is', etc.
    min_df       = 5           # ignore terms appearing in fewer than 5 docs
)
tfidf_train = tfidf.fit_transform(X_train['description'].fillna(''))
tfidf_test  = tfidf.transform(X_test['description'].fillna(''))

# Convert to DataFrame and join back
tfidf_cols = [f'tfidf_{w}' for w in tfidf.get_feature_names_out()]
tfidf_train_df = pd.DataFrame(tfidf_train.toarray(), columns=tfidf_cols, index=X_train.index)
X_train = pd.concat([X_train, tfidf_train_df], axis=1)

# ── Bag of Words (CountVectorizer) ───────────────────────────────────────────
# Use when: raw term counts are more useful than TF-IDF weights
# Less common than TF-IDF but useful when document length is consistent
cv = CountVectorizer(max_features=50, stop_words='english')
bow_train = cv.fit_transform(X_train['description'].fillna(''))

# ── Category from text — keyword-based labeling ───────────────────────────────
# Use when: a few keywords cleanly define the category
keyword_map = {
    'electronics': ['phone', 'laptop', 'tablet', 'camera'],
    'clothing':    ['shirt', 'pants', 'dress', 'shoes'],
    'food':        ['snack', 'drink', 'meal', 'coffee'],
}
def classify_by_keywords(text, keyword_map):
    text = str(text).lower()
    for category, keywords in keyword_map.items():
        if any(kw in text for kw in keywords):
            return category
    return 'other'

df['product_category'] = df['description'].apply(classify_by_keywords, keyword_map=keyword_map)

**Common mistakes:**
- Fitting `TfidfVectorizer` on the full dataset — vocabulary and IDF weights leak test distribution; always fit on train only
- Not filling NaN before vectorizing — `TfidfVectorizer` raises an error on NaN values; use `.fillna('')`
- Using too many `max_features` — 100–500 is usually enough; more features add noise and memory pressure

---
## §8 — Leakage & Validation

Feature leakage is when information from the future (or from the target itself) is encoded into features used to train the model. It produces unrealistically high training performance and catastrophic failure in production.

In [ ]:
# ── Type 1: Target leakage — feature is computed using the target ─────────────
# Example: predicting loan default, and a feature is 'days_past_due'
# days_past_due is 0 for non-defaulters and >0 for defaulters — it IS the target

# Detection: check correlation of each feature with the target
correlations = X_train.corrwith(y_train).abs().sort_values(ascending=False)
print(correlations.head(10))
# Suspicious: correlation > 0.8 with the target → investigate that feature

# Detection: check if any feature is derived from post-event data
# Ask for each feature: "Would I know this value at prediction time?"
# e.g. 'refund_amount' for a churn model — you only know the refund AFTER churn happens

# ── Type 2: Temporal leakage — using future data for a past prediction ────────
# Wrong: random split on time-series data
from sklearn.model_selection import train_test_split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
# This mixes future dates into training — lag features for training rows
# can look up rows that are chronologically later

# Correct: time-based split
cutoff = df['date'].quantile(0.8)   # or a fixed business date
train  = df[df['date'] <= cutoff]
test   = df[df['date'] >  cutoff]

# ── Type 3: Group leakage — same entity in train and test ─────────────────────
# Example: predicting customer LTV; same customer_id in both train and test
# The model memorizes individual customers rather than learning patterns

# Correct: group-aware split (all rows of one entity go to one split)
from sklearn.model_selection import GroupShuffleSplit
gss = GroupShuffleSplit(n_splits=1, test_size=0.2, random_state=42)
train_idx, test_idx = next(gss.split(X, y, groups=df['user_id']))
X_train, X_test = X.iloc[train_idx], X.iloc[test_idx]

# ── Leakage diagnosis checklist ──────────────────────────────────────────────
# 1. Model accuracy is suspiciously high (>95% on a hard problem)?
# 2. Performance drops dramatically from validation to production?
# 3. Feature importance shows an unexpected column dominating?
# 4. Features computed after the prediction point (post-event data)?
# 5. Rolling/window features that didn't use .shift(1) before .rolling()?
# 6. Scalers, encoders, or imputers fit on the full dataset instead of train only?

# ── Safe preprocessing pipeline — fit on train, transform both ───────────────
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.impute import SimpleImputer

pipeline = Pipeline([
    ('impute', SimpleImputer(strategy='median')),
    ('scale',  StandardScaler())
])
X_train_processed = pipeline.fit_transform(X_train)   # fit + transform on train
X_test_processed  = pipeline.transform(X_test)        # transform only on test — no refit

**Leakage types and fixes:**

| Type | Signal | Fix |
| :--- | :--- | :--- |
| Target leakage | Feature correlates >0.8 with target; comes from post-event data | Remove the feature; ask "would I know this at prediction time?" |
| Temporal leakage | Random split on time-series; rolling features without `shift(1)` | Time-based split; always `shift(1)` before `.rolling()` |
| Group leakage | Same entity ID in train and test | `GroupShuffleSplit` or entity-level split |
| Preprocessing leakage | Scaler/encoder fit on full data before split | Use `sklearn.Pipeline`; fit only on train |
| Target encoding leakage | Category means computed without CV | Use out-of-fold CV target encoding (see §1) |

**Common mistakes:**
- Treating suspiciously high accuracy as a win — if a churn model hits 99% AUC, check for leakage before celebrating
- Filling missing values before splitting — the imputed value (e.g. mean) is computed from the full dataset, leaking test statistics into train
- Using `sklearn` transformers outside a `Pipeline` and manually fitting on all data — the `Pipeline` pattern enforces correct fit-on-train behavior

---
## Decision Guide

```
Encoding a categorical column?
├── Nominal, <15 unique values, linear model    → one-hot (drop_first=True)           (§1)
├── Ordinal (Low/Med/High)                      → label/ordinal encoding               (§1)
├── High cardinality, tree model                → frequency encoding                   (§1)
└── High cardinality, strong target signal      → target encoding with CV              (§1)

Transforming a numeric column?
├── Right-skewed (revenue, counts)              → np.log1p()                           (§2)
├── Linear model, SVM, KNN, NN                 → StandardScaler (fit on train only)   (§2)
├── Has significant outliers                    → RobustScaler                         (§2)
└── Want auto-normalization                     → PowerTransformer(Yeo-Johnson)        (§2)

Binning a continuous variable?
├── Business-defined thresholds                 → pd.cut()                             (§3)
├── Equal-sized groups                          → pd.qcut() on train only              (§3)
└── Multi-condition business logic              → np.select()                          (§3)

Extracting from a datetime column?
├── Cyclical component (hour, month, weekday)   → sin/cos encoding                    (§4)
├── Time elapsed since an event                 → (ts - ref_date).dt.days             (§4)
└── Time since last event per user              → groupby().shift(1) → diff in hours  (§4)

Creating lag/window features?
├── Previous value                              → groupby().shift(1)                   (§5)
├── Rolling mean/std over last N rows           → shift(1).rolling(N).mean()          (§5)
├── Cumulative sum from start of series         → shift(1).expanding().sum()          (§5)
└── Train/test split                            → time-based cutoff, NOT random       (§5)

Combining two columns?
├── One normalizes the other (CTR, ARPU)        → ratio feature (watch for /0)        (§6)
├── Gap from a reference (vs group mean)        → difference feature                  (§6)
└── Both features interact                      → product feature                     (§6)

Text column in a tabular dataset?
├── Simple signal (length, keywords, counts)    → str.len(), str.contains()           (§7)
└── Richer vocabulary signal                    → TfidfVectorizer (fit on train only) (§7)

Suspecting leakage?
├── Model accuracy is unrealistically high      → check feature correlations with target (§8)
├── Time-series with random split               → switch to time-based split           (§8)
├── Same user in train and test                 → GroupShuffleSplit                    (§8)
└── Preprocessing fit on full data              → use sklearn.Pipeline                 (§8)
```